### Test Pydantic AI with LangChain's SQL tools

In [1]:
from pydantic_ai import Agent, RunContext
from dotenv import load_dotenv
import psycopg
import nest_asyncio
import os

load_dotenv('../.env')
neon_conn_string = os.getenv("NEON_DB_URL")
langchain_neon_conn_string = neon_conn_string.replace('postgresql', 'postgresql+psycopg')
nest_asyncio.apply() # for async issues in Jupyter stuff

Get the enums

In [2]:
enums = ['availability_status_type', 'room_bed_type', 'room_status_type', 'room_type']
enum_values = {}

with psycopg.connect(neon_conn_string) as neon_conn:
    with neon_conn.cursor() as cur:
        for enum in enums:
            cur.execute(f"SELECT enum_range(NULL::{enum})")
            enum_values[enum] = [val.strip('\'"{}') for val in cur.fetchall()[0][0].split(',')]
        cur.execute("SELECT DISTINCT unnest(basic_amenities) FROM rooms;")
        basic_amenities = [val[0] for val in cur.fetchall()]
        cur.execute("SELECT DISTINCT unnest(additional_amenities) FROM rooms;")
        additional_amenities = [val[0] for val in cur.fetchall()]
        cur.execute("SELECT DISTINCT unnest(view_type) FROM rooms;")
        view_types = [val[0] for val in cur.fetchall()]
        
            
print(f"enums = {enum_values}")
print(f"basic amenities = {basic_amenities}")
print(f"additional amenities = {additional_amenities}")
print(f"view types = {view_types}")

enums = {'availability_status_type': ['Booked', 'Available', 'Maintenance'], 'room_bed_type': ['Queen', 'Double Queen', 'King', 'Double King', 'King + Sofa Bed', 'King + Multiple Sofa Beds'], 'room_status_type': ['Available', 'Occupied', 'Maintenance'], 'room_type': ['Standard', 'Deluxe', 'Suite', 'Presidential Suite']}
basic amenities = ['Full Kitchen', 'Executive Office', 'Nespresso Machine', 'Premium Coffee Maker', 'High-Speed WiFi', 'Premium Bathrobes', 'Kitchenette', 'Bathrobes', "Butler's Pantry", 'Welcome Amenity', 'Full-Size Refrigerator', '55" Smart TV', 'Bluetooth Speaker', 'Professional Coffee Bar', 'Guest Bathroom', 'Air Conditioning', 'Living Room', 'Luxury Welcome Amenity', 'Bang & Olufsen Sound System', 'Ultra-High-Speed WiFi', 'Work Desk', 'Multiple 75" Smart TVs', 'In-Room Safe', 'Smart TV', 'Living Room Area', 'Personalized Stationery', 'Hair Dryer', 'Walk-in Closet', 'Multiple Bathrooms', 'Dining Area', 'Bose Sound System', '65" Smart TV', 'Slippers', 'Evening Turndo

In [3]:
from langchain_community.utilities import SQLDatabase

schema_description = {
    "rooms": (f"""
            CREATE TABLE rooms (
                room_id INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
                room_number INT NOT NULL,
                floor INT NOT NULL,
                type room_type, -- Enum with options {enum_values['room_type']}
                square_feet INT,
                basic_amenities TEXT[],  -- Options are {basic_amenities}
                additional_amenities TEXT[], -- Options are {additional_amenities}
                max_occupancy INT,
                bed_type room_bed_type,  -- Enum with options {enum_values['room_bed_type']}
                view_type TEXT[],  -- Options are {view_types}
                accessibility BOOLEAN,  -- Whether handicapped accessible
                status room_status_type, -- Enum with options {enum_values['room_status_type']}
                last_renovation DATE,
                base_rate NUMERIC(10, 2),  -- Do not provide unless asked for
                max_rate NUMERIC(10, 2)  -- Do not provide unless asked for
                -- The table lists details about all the rooms in the hotel.
            );
    """),
    "room_availability": (f"""
            CREATE TABLE room_availability (
                id INT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
                room_id INT NOT NULL,
                room_number INT NOT NULL,
                date DATE NOT NULL,
                status availability_status_type,  -- Enum with options {enum_values['availability_status_type']}
                price NUMERIC(8,2),
                max_occupancy INT,
                FOREIGN KEY (room_id) REFERENCES rooms(room_id)
                -- The table lists the room availability by date and the corresponding rate
            );
    """),
}

db = SQLDatabase.from_uri(database_uri=langchain_neon_conn_string, include_tables=schema_description.keys(), custom_table_info=schema_description)